# Fightin' words + Ngram

Associated chart: the fightin-words scrolly and the ngram search

Output files:
`fighting_words_grid.json`, `fighting_words_examples.json`, `fw_examples_lit.json`,
`fox_kw.json`, and `fox_ngrams.csv` in `src/data/`.

Input files: `data/fox_wnba.csv`, `data/just_womens_sports.csv`, `data/yahoo_sports.csv`, `data/espn.csv`, `data/cbs_sports.csv`, `data/usa_today.csv`.

Requirements: scikit-learn (~15 minutes)

`pip install scikit-learn`


## Setup

In [ ]:
import csv, json, re
from collections import Counter
from pathlib import Path
import pandas as pd
from fighting_words import bayes_compare_language

In [ ]:
DATA, OUT = Path("../data"), Path("../src/data")
csv.field_size_limit(10 ** 9)

MATCHED_FROM = "2023-04-01"   # the window where all the archives are dense
BODY_WORDS = 400              # words kept per article, so one long feature can't dominate
MIN_ARTICLES = 300
MIN_PLAYER_HEADLINES = 30

# the same five outlets the rest of the piece sets Fox against (Yahoo, mostly
# syndicated wire copy, is out); The Athletic's bodies come from harvest_athletic_bodies.py
GROUPS = {
    # the five comparison outlets, together and one at a time (Yahoo, mostly
    # syndicated wire copy, is out); The Athletic's bodies come from harvest_athletic_bodies.py
    "all outlets": ["ESPN", "CBS Sports", "USA Today", "Just Women's Sports", "The Athletic"],
    "ESPN": ["ESPN"],
    "CBS Sports": ["CBS Sports"],
    "USA Today": ["USA Today"],
    "The Athletic": ["The Athletic"],
    "Just Women's Sports": ["Just Women's Sports"],
}

FILES = [("fox_wnba", "Fox News"), ("espn", "ESPN"), ("cbs_sports", "CBS Sports"),
         ("usa_today", "USA Today"), ("just_womens_sports", "Just Women's Sports"),
         ("the_athletic", "The Athletic")]


In [ ]:
corpora, bodies = {}, {}   # outlet -> [(date, headline)] / [(date, first 400 words)]
for name, outlet in FILES:
    heads, docs, seen = [], [], set()

    for row in csv.DictReader((DATA / f"{name}.csv").open()):
        # date
        date = (row.get("date") or "")[:10]

        # headline
        headline = (row.get("headline") or "").strip()

        # body
        body = (row.get("body") or "").strip()
        if date and headline and not (outlet == "Fox News" and headline in seen):
            seen.add(headline)
            heads.append((date, headline))
        if date and len(body.split()) > 80:
            docs.append((date, " ".join(body.split()[:BODY_WORDS])))

    corpora[outlet] = heads
    if len(docs) >= MIN_ARTICLES:   # CBS and USA Today are headline-only harvests
        bodies[outlet] = docs

matched = lambda rows: [r for r in rows if r[0] >= MATCHED_FROM]


In [ ]:
corpora

In [ ]:
corpora['Fox News'][0]

In [ ]:
bodies['Fox News'][0]

In [ ]:
pd.DataFrame([(o, len(rows), len(matched(rows)), len(matched(bodies.get(o, []))))
              for o, rows in corpora.items()],
             columns=["outlet", "headlines", "matched", "matched bodies"])

## Cleaning

Remove stopwords and boilerplate. Convert to lowercase. Merge player names.

In [ ]:
NODES = json.loads((OUT / "headline_network.json").read_text())["nodes"]
DISPLAY = {re.sub(r"[^a-z]+", "", n["name"].lower()): n["name"] for n in NODES}

def name_forms(name):
    parts = name.lower().split()
    surname = re.sub(r"[^a-z]", "", parts[-1])
    first = re.sub(r"[^a-z]", "", parts[0])
    variants = {name.lower(), surname, f"{surname}'s", f"{surname}s"}
    if len(first) > 3:
        variants.add(first)
    return sorted(variants, key=len, reverse=True)

PEOPLE = {token: name_forms(name) for token, name in
          ((re.sub(r"[^a-z]+", "", n["name"].lower()), n["name"]) for n in NODES)}

# longest form first so "caitlin clark" is matched before bare "clark"
NAME_PATTERNS = sorted(((re.compile(r"\b" + re.escape(f) + r"\b", re.I), token)
                        for token, fs in PEOPLE.items() for f in fs),
                       key=lambda pair: -len(pair[0].pattern))
                       
POSSESSIVE = re.compile(r"\b(" + "|".join(PEOPLE) + r")(?:'s|’s|s)\b")

STOPWORDS = set("""a an the and or but if then than that this these those of in on at to for with
from by as is are was were be been being it its it's his her she he they them their our your you
i we us not no do does did has have had will would can could should may might must about after
before over under again more most other some such only own same so too very s t don now new
what which who whom when where why how all any both each few nor own said says say
re ve ll didn don doesn isn wasn aren won wouldn couldn shouldn hasn haven hadn shes hes
theyre youre weve ive got get gets getting just also even still back one two per like""".split())

BOILERPLATE = re.compile(
    r"click here to get the fox news app|get the fox news app|"
    r"follow (?:us|fox news digital)[^.]*\.|fox news digital|"
    r"sign up for[^.]*newsletter[^.]*\.|subscribe to[^.]*\.|"
    r"download the (?:espn|yahoo|cbs) app[^.]*\.|"
    r"this article (?:first )?appeared on[^.]*\.|"
    r"\(?(?:ap|reuters|getty images)\)?\s*[-—]|all rights reserved|"
    r"copyright \d{4}[^.]*\.", re.I)

BODY_STOPWORDS = {"fox", "espn", "yahoo", "cbs", "usa", "today", "news", "com", "app",
                  "click", "here", "follow", "digital", "newsletter", "subscribe",
                  "images", "getty", "related", "appeared", "story", "video", "watch",
                  "read", "share", "photo", "via", "said", "says",
                  "outkick", "foxnews", "coverage", "download", "associated", "press",
                  "reuters", "contributed", "report", "reporter", "correspondent"}

def clean(headline):
    text = headline.lower()
    for pattern, token in NAME_PATTERNS:
        text = pattern.sub(token, text)
    text = POSSESSIVE.sub(r"\1", text)          # "caitlinclark's" -> "caitlinclark"
    text = text.replace("'", "").replace("’", "")
    return " ".join(w for w in re.findall(r"[a-z]{3,}", text) if w not in STOPWORDS)

def clean_body(text):
    return " ".join(w for w in clean(BOILERPLATE.sub(" ", text)).split()
                    if w not in BODY_STOPWORDS)

head_m = {o: matched(rows) for o, rows in corpora.items()}
body_m = {o: matched(rows) for o, rows in bodies.items()}
head_clean = {o: [clean(h) for _, h in rows] for o, rows in head_m.items()}
body_clean = {o: [clean_body(t) for _, t in rows] for o, rows in body_m.items()}


In [ ]:
head_clean

In [ ]:
head_clean.keys()

In [ ]:
body_clean

In [ ]:
body_clean.keys()

In [ ]:
print({o: len(v) for o, v in head_clean.items()}, "headlines cleaned;",
      sum(map(len, body_clean.values())), "bodies cleaned")

In [ ]:
!pip3 install convokit

In [ ]:
!pip3 install convokit

In [ ]:
bayes_compare_language(body_clean['Fox News'], body_clean['ESPN'])

In [ ]:
all_other_outlets = [d for o in GROUPS['all outlets'] if o in body_clean for d in body_clean[o]]

In [ ]:
all_other_outlets

## Fox News vs All

Top 30 all words 

In [ ]:
bayes_compare_language(body_clean['Fox News'], all_other_outlets)[:30]

Top 30 Fox words

In [ ]:
bayes_compare_language(body_clean['Fox News'], all_other_outlets)[-30:]

In [ ]:
fightin_words = bayes_compare_language(body_clean['Fox News'], all_other_outlets)

In [ ]:
fightin_df = pd.DataFrame(fightin_words, columns=['word', 'z_score'])

In [ ]:
fightin_df['direction'] = fightin_df['z_score'].apply(lambda x: 'Fox News' if x > 0 else 'All')

In [ ]:
fightin_df

In [ ]:
def plot_fightin_words(fightin_df, top_n=15):
    fig, ax1 = plt.subplots(figsize=(6, 8))

    # Sort words by z-score
    fightin_df = fightin_df.sort_values(by='z_score', ascending=True)

    # Select top words from each group
    gpt_labels = fightin_df[fightin_df['z_score'] < 0].head(top_n)
    poetry_labels = fightin_df[fightin_df['z_score'] > 0].tail(top_n)

    # Combine selected words into one DataFrame
    fightin_df = pd.concat([gpt_labels, poetry_labels])

    # Create the main barplot
    sns.barplot(data=fightin_df, x='z_score', y='word', hue='direction', 
               palette=sns.color_palette([
         "#33BBEE", "#4477AA",
    ], 2), 
                dodge=False, ax=ax1)

    plt.title(f"All Outlets vs. Fox News", fontsize=16, fontweight='bold')
    ax1.set_xlabel('Z-Score (Distinctiveness)')
    ax1.set_ylabel('Most Distinctive Words\n', fontsize=14,fontweight='bold')


    # Create a second y-axis for the right side (Poetry Foundation labels)
    ax2 = ax1.twinx()

    # Manually set the y-ticks for both sides to match the words on each side
    y_ticks_gpt = np.arange(len(gpt_labels))  # y-ticks for GPT
    y_ticks_poetry = np.arange(len(poetry_labels)) + len(gpt_labels)  # Adjust y-ticks for Poetry
    ax1.set_yticks(y_ticks_gpt)
    ax1.set_yticklabels(gpt_labels['word'], fontsize=12,
    # fontweight='bold', 
                        #color='#4477AA'
                        )

    # Erase unused ticks/labels

    # Create a list of blank values with the same length as gpt_labels
    blank_values = [''] * len(gpt_labels['word'])

    poetry_labels_with_blanks = poetry_labels['word'].tolist()  # Convert to list
    poetry_labels_with_blanks.reverse()  # Reverse in place
    poetry_labels_with_blanks += blank_values  # Append blank values

    ax2.set_yticks(np.arange(len(poetry_labels_with_blanks)))  # Ensure y-ticks are numeric
    ax2.set_yticklabels(poetry_labels_with_blanks, fontsize=12, 
                        #fontweight='bold',
                         #color='#33BBEE'
                         )
    
    # Make the first 5 ticks on the twin axis invisible
    ax2_ticklines = ax2.get_yticklines()[::-1]  # Reverse the list

    # Hide tick marks 
    for i in range(top_n * 2):
        #ax2_ticks[i].set_visible(False)  
        ax2_ticklines[i].set_visible(False)  # Hide tick marks

    # Add a horizontal line separating GPT and Poetry Foundation words
    plt.axhline(y=len(gpt_labels) - 0.5, color='black', linestyle='--', linewidth=1)

    # Adjust the legend position to the top-right outside the plot
    ax1.legend(title='Source',
                loc='lower left', 
                #bbox_to_anchor=(1.55, 1)
                )

    plt.tight_layout()

    # plt.savefig('fightin-words-gpt4.png', dpi=300)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
plot_fightin_words(fightin_df)

Fox News vs ESPN

In [ ]:
bayes_compare_language(body_clean['Fox News'], body_clean['ESPN'])[:30]

In [ ]:
bayes_compare_language(body_clean['Fox News'], body_clean['ESPN'])[-30:]

## Fightin words

Headlines, whole articles (Yahoo and ESPN are the outlets we have full text for), and headlines naming Caitlin Clark. 



In [ ]:
def record(scored, fox_docs, other_docs, group, bucket, floor):

    fox_counts, other_counts = Counter(), Counter()
   
    for doc in fox_docs:
        fox_counts.update(doc.split())
    
    for doc in other_docs:
        other_counts.update(doc.split())
   
    for word, z in scored:
        total = fox_counts[word] + other_counts[word]
        if total < floor:
            continue
        row = bucket.setdefault(DISPLAY.get(word, word), {"n": {}, "z": {}, "nf": {}, "no": {}})
        row["z"][group] = round(float(z), 3)
        row["n"][group] = total
        row["nf"][group] = fox_counts[word]
        row["no"][group] = other_counts[word]

def compare(fox_docs, other_docs, group, bucket, floor):
    
    scored = bayes_compare_language(fox_docs, other_docs, ngram=1, prior=0.05)
    record(scored, fox_docs, other_docs, group, bucket, floor)

grid, group_counts = {}, {}

for group, members in GROUPS.items():
    other = [d for o in members for d in head_clean[o]]
    compare(head_clean["Fox News"], other, group, grid, floor=5)
    group_counts[group] = {"fox": len(head_clean["Fox News"]), "other": len(other)}

# article text: CBS + USA Today have no bodies, so "all outlets" == "Yahoo + ESPN"
grid_bodies, body_counts, cache = {}, {}, {}
for group, members in GROUPS.items():
    present = tuple(o for o in members if o in body_clean)
    other = [d for o in present for d in body_clean[o]]
    if len(other) < MIN_ARTICLES:
        continue
    if present not in cache:
        cache[present] = bayes_compare_language(body_clean["Fox News"], other, ngram=1, prior=0.05)
    record(cache[present], body_clean["Fox News"], other, group, grid_bodies, floor=3)
    body_counts[group] = {"fox": len(body_clean["Fox News"]), "other": len(other)}

by_player, player_counts = {}, {}

for player in ["Caitlin Clark"]:
    surname = player.split()[-1].lower()
    def about(outlet):
        return [c for (d, h), c in zip(head_m[outlet], head_clean[outlet])
                if surname in h.lower()]
    fox_rows = about("Fox News")
    if len(fox_rows) < MIN_PLAYER_HEADLINES:
        continue
    bucket, counts = {}, {}
    for group, members in GROUPS.items():
        other = [d for o in members for d in about(o)]
        if len(other) < MIN_PLAYER_HEADLINES:
            continue
        compare(fox_rows, other, group, bucket, floor=3)
        counts[group] = {"fox": len(fox_rows), "other": len(other)}
    words = [{"word": w, "z": r["z"], "n": r["n"], "nf": r["nf"], "no": r["no"]}
             for w, r in bucket.items() if r["z"]]
    if words:
        by_player[player] = words
        player_counts[player] = counts

serialise = lambda bucket: [{"word": w, "z": r["z"], "n": r["n"], "nf": r["nf"], "no": r["no"]}
                            for w, r in bucket.items() if r["z"]]

grid_words, body_words = serialise(grid), serialise(grid_bodies)

#### Overwrites `src/data/fighting_words_grid.json`

In [ ]:
fw_grid = ({
    "outlets": [g for g in GROUPS if g in group_counts],
    "outletN": group_counts,
    "words": grid_words,
    "articleOutlets": [g for g in GROUPS if g in body_counts],
    "articleOutletN": body_counts,
    "articleWords": body_words,
    "players": sorted(by_player),
    "byPlayer": by_player,
    "playerN": player_counts,
    "note": "weighted log-odds z-scores and word frequencies, "
            "matched 2023-2026 window, headlines only",
})
print(f"grid: {len(grid_words):,} headline words, {len(body_words):,} article words, "
      f"{len(by_player)} player")

In [ ]:
z = (pd.DataFrame([(w["word"], w["z"].get("all outlets"), w["n"].get("all outlets"))
                   for w in grid_words], columns=["word", "z", "n"])
        .dropna().sort_values("z"))

pd.concat([z.tail(12).iloc[::-1].reset_index(drop=True).add_prefix("fox_"),
           z.head(12).reset_index(drop=True).add_prefix("other_")], axis=1)

In [ ]:
z

## Example headlines and sentences

One example per grid word for the hover card, pulled from the side the word leans toward. Exact word boundary first; loose prefix only as a fallback, so "trans" finds *trans*, not *transferred*. Kept out of the grid file on purpose — it is ~3MB and loads lazily.

In [ ]:
SHOUT_RUN = re.compile(r"\b(?:[A-Z][A-Z'’.&-]{1,}[,:;?!]?\s+){3,}[A-Z][A-Z'’.&-]{1,}[,:;?!]?")

def patterns(word):
    return [re.compile(rf"\b({re.escape(word)})\b", re.I),
            re.compile(rf"\b({re.escape(word)}\w*)", re.I)]

def lean(entry):
    zs = list(entry["z"].values())
    return sum(zs) / max(1, len(zs))

# pools carry a lowercased copy so a cheap substring test can skip most rows
def pool(rows):
    return [(text, text.lower()) for _, text in rows]

def headline_example(word, rows):
    lowered = word.lower()
    for pattern in patterns(word):
        for text, low in rows:
            if lowered not in low:
                continue
            match = pattern.search(text)
            if match:
                return [text.strip()[:200], match.group(1)]
    return None

def sentence_example(word, rows):
    lowered = word.lower()
    for pattern in patterns(word):
        for body, low in rows:
            if lowered not in low:
                continue
            match = pattern.search(body)
            if not match:
                continue
            # the sentence around the first match, cleaned of spliced ALL-CAPS links
            for candidate in re.split(r"(?<=[.!?])\s+", body.replace("\n", " ")):
                m2 = pattern.search(candidate)
                if not m2:
                    continue
                candidate = re.sub(r"\s{2,}", " ",
                                   SHOUT_RUN.sub(" ", candidate)).strip(" —-–,;:")
                # the match must survive the cleanup: a word found only in a
                # spliced ALL-CAPS link is gone now, and the sentence would
                # not contain it
                m3 = pattern.search(candidate)
                if m3 and 30 <= len(candidate) <= 320:
                    return [candidate, m3.group(1)]
            # no clean sentence in this article carries the word (it was only in
            # a spliced link, or the sentence ran too long): try the next one
            continue
    return None

def build(entries, fox_rows, other_rows, finder):
    out, missing = {}, 0
    for word, entry in entries.items():
        first, second = (fox_rows, other_rows) if lean(entry) >= 0 else (other_rows, fox_rows)
        example = finder(word, first) or finder(word, second)
        if example:
            out[word] = example
        else:
            missing += 1
    return out, missing

fox_heads = pool(head_m["Fox News"])
other_heads = pool([r for o, rows in head_m.items() if o != "Fox News" for r in rows])
fox_bodies = pool(body_m.get("Fox News", []))
other_bodies = pool([r for o, rows in body_m.items() if o != "Fox News" for r in rows])

# words and byPlayer both draw from headlines; articleWords from full text
head_entries = {e["word"]: e for e in grid_words}
for player_words in by_player.values():
    for e in player_words:
        head_entries.setdefault(e["word"], e)
heads_ex, miss_h = build(head_entries, fox_heads, other_heads, headline_example)
arts_ex, miss_a = build({e["word"]: e for e in body_words},
                        fox_bodies, other_bodies, sentence_example)

#### Overwrites `src/data/fighting_words_examples.json`

In [ ]:
(OUT / "fighting_words_examples.json").write_text(
    json.dumps({"headlines": heads_ex, "articles": arts_ex}, ensure_ascii=False))
print(f"examples: {len(heads_ex):,} headline ({miss_h} missing), "
      f"{len(arts_ex):,} article ({miss_a} missing)")

In [ ]:
LIT = ["hard", "foul", "fouls", "calls", "flagrant", "race", "racism", "racist",
       "privilege", "black", "white", "trans", "transgender", "womens", "media",
       "star", "fans", "social", "olympics", "alleged", "Donald Trump", "Caitlin Clark"]

src = json.loads((OUT / "fighting_words_examples.json").read_text())
lit = {mode: {w: src[mode][w] for w in LIT if w in src[mode]}
       for mode in ("headlines", "articles")}

#### Overwrites `src/data/fw_examples_lit.json`

In [ ]:
(OUT / "fighting_words.json").write_text(json.dumps(
    {"grid": fw_grid, "lit": lit}, ensure_ascii=False))
print(f"lit examples: {len(lit['headlines'])} headline / {len(lit['articles'])} article words")

## Fox ngram series and search index

`fox_ngrams.csv`: per month, how many Fox WNBA articles mention each curated word. "white" is matched lowercase only — case-insensitive would mostly count Fever coach Stephanie White. `fox_kw.json`: the same monthly counts for every word and 2-3 word phrase appearing in 10+ articles, for the chart's search box.

In [ ]:
NGRAM_WORDS = [
    ("trans", re.compile(r"\btrans\b", re.I)),
    ("transgender", re.compile(r"\btransgender\b", re.I)),
    ("protect", re.compile(r"\bprotect", re.I)),
    ("race", re.compile(r"\brace\b", re.I)),
    ("white", re.compile(r"\bwhite\b")),   # lowercase only
    ("trans language", re.compile(
        r"\btrans\b|\btransgender\b|\bbiological (?:males?|m[ae]n)\b", re.I)),
    ("caitlin clark", re.compile(r"caitlin clark|\bcaitlin\b", re.I)),
    ("sophie cunningham", re.compile(r"sophie cunningham|\bcunningham\b", re.I)),
]

fox_rows = list(csv.DictReader((DATA / "fox_wnba.csv").open()))
totals = Counter()
hits = {w: Counter() for w, _ in NGRAM_WORDS}
for row in fox_rows:
    month = (row.get("date") or "")[:7]
    if len(month) < 7:
        continue
    totals[month] += 1
    text = (row.get("headline") or "") + " " + (row.get("body") or "")
    for word, pattern in NGRAM_WORDS:
        if pattern.search(text):
            hits[word][month] += 1

months = sorted(totals)

#### Overwrites `src/data/fox_ngrams.csv`

In [ ]:
with (OUT / "fox_ngrams.csv").open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["month", "articles"] + [w for w, _ in NGRAM_WORDS])
    for m in months:
        writer.writerow([m, totals[m]] + [hits[w].get(m, 0) for w, _ in NGRAM_WORDS])
print(f"fox_ngrams.csv: {len(months)} months ({months[0]}..{months[-1]}), "
      f"{sum(totals.values()):,} articles")

In [ ]:
KW_BASE_STOP = set("""a an the and or but of in on at to for with by from as is are was were be been
being it its this that these those i you he she they we his her their our your my me him them
us new says said say after amid over vs vs. into out up down off not no what who why how when
where which while will would could should can may might must have has had do does did done
about more most so than then just also only even still back get gets got make makes made
""".split())
KW_STOP = KW_BASE_STOP | {"wnba", "basketball", "game", "games", "team", "teams", "season",
                          "league", "player", "players", "coach", "points", "click", "here",
                          "fox", "news", "digital", "coverage", "com", "www", "https", "http"}

def kw_tokenize(text):
    return re.findall(r"[a-z][a-z-]+", re.sub(r"[‘’']s\b", "", text.lower()))

from collections import defaultdict
kw_month = defaultdict(lambda: defaultdict(int))
kw_total, kw_ex = Counter(), {}
ph_month = defaultdict(lambda: defaultdict(int))
ph_total, ph_ex = Counter(), {}
for row in fox_rows:
    mo = (row.get("date") or "")[:7]
    if len(mo) < 7 or mo < "2019":
        continue
    ts = kw_tokenize((row.get("headline") or "") + " " + (row.get("body") or ""))
    title = row.get("headline") or ""
    for t in set(ts):
        if t in KW_STOP or len(t) < 3:
            continue
        kw_month[t][mo] += 1
        kw_total[t] += 1
        kw_ex.setdefault(t, title)
    grams = set()   # 2-3 word phrases, content-word edges only
    for n in (2, 3):
        for i in range(len(ts) - n + 1):
            parts = ts[i:i + n]
            if parts[0] in KW_BASE_STOP or parts[-1] in KW_BASE_STOP:
                continue
            grams.add(" ".join(parts))
    for g in grams:
        ph_month[g][mo] += 1
        ph_total[g] += 1
        ph_ex.setdefault(g, title)

kw_out = {w: dict(kw_month[w]) for w, n in kw_total.items() if n >= 10}
for g, n in ph_total.items():
    if n >= 10:
        kw_out[g] = dict(ph_month[g])
        kw_ex[g] = ph_ex.get(g, "")

#### Overwrites `src/data/fox_kw.json`

In [ ]:
(OUT / "fox_kw.json").write_text(json.dumps(
    {"min_articles": 10, "words": kw_out,
     "examples": {w: kw_ex.get(w, "") for w in kw_out}}))
print(f"fox_kw.json: {len(kw_out):,} words/phrases, "
      f"{(OUT / 'fox_kw.json').stat().st_size // 1024} KB")

## Spot check

In [ ]:
import random

grid = fw_grid
entries = [{"word": e["word"], "z": e["z"]["all outlets"]} for e in grid["words"]
           if "all outlets" in e["z"]]
fox_side = [e for e in entries if e["z"] > 1.96]
other_side = [e for e in entries if e["z"] < -1.96]
fox_pool = [h for _, h in head_m["Fox News"]]
other_pool = [h for o, rows in head_m.items() if o != "Fox News" for _, h in rows]

def show(side, pool, label):
    print(f"— random {label} words, with real headlines that contain them")
    for e in random.sample(side, min(3, len(side))):
        hits = [h for h in pool if e["word"] in h.lower()]
        print(f"   {e['word']!r} (z={e['z']:+.1f}, {len(hits)} headlines)")
        for h in random.sample(hits, min(2, len(hits))):
            print(f"      {h[:82]}")

show(fox_side, fox_pool, "Fox-side")
show(other_side, other_pool, "other-side")